# 02 — Training (restart-safe)

**But** : entraîner le modèle ResNet50 sur PatchCamelyon.

Ce notebook est **restart-safe** :
- Il cherche automatiquement le dernier checkpoint dans le dossier de sortie.
- S'il en trouve un, il reprend l'entraînement là où il s'était arrêté (optimizer, epoch, historique).
- Sinon, il part de zéro.

> 💡 Si Google Colab coupe le runtime, relancez simplement ce notebook du début : il reprendra tout seul.

## 1. Setup

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/LAAFI_AI-main
except ImportError:
    pass



Mounted at /content/drive
/content/drive/MyDrive/LAAFI_AI-main


In [3]:
%%capture
%pip install -q -r requirements-colab.txt

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [5]:
import datasets
datasets.logging.set_verbosity_error()

from laafi_ai.cli_train import set_seed
from laafi_ai.config import ExperimentConfig
from laafi_ai.data import PCamDataModule
from laafi_ai.logging_utils import setup_logging
from laafi_ai.model import build_resnet50_classifier, count_trainable_parameters, get_device
from laafi_ai.paths import resolve_project_paths
from laafi_ai.trainer import Trainer

setup_logging()

## 2. Configuration

Modifiez les lignes ci-dessous pour passer du mode *smoke test* à un entraînement complet.

In [6]:
config = ExperimentConfig.from_yaml('configs/default.yaml')

# =====================================================================
# SMOKE TEST vs ENTRAÎNEMENT COMPLET
# =====================================================================
# Pour un smoke test rapide, décommentez les deux lignes ci-dessous :
config.data.max_train_samples = None
config.data.max_val_samples = None
#
# Pour un entraînement complet, laissez-les commentées (None = tout).
# =====================================================================

config.data.batch_size = 32
config.training.epochs = 10
config.model.freeze_backbone = True
config.model.unfreeze_layer4 = True
config.optimizer.learning_rate = 1e-4
config.output_dir = 'outputs_finetune_layer4'

set_seed(config.seed)
device = get_device(config.device)
paths = resolve_project_paths(config, project_root=PROJECT_ROOT)

print('Device:', device)
print('Epochs:', config.training.epochs)
print('Output:', paths.output_dir)

2026-07-06 23:43:30,388 | INFO | laafi_ai.paths | Project root: /content/drive/MyDrive/LAAFI_AI-main
2026-07-06 23:43:30,390 | INFO | laafi_ai.paths | Output directory: /content/drive/MyDrive/LAAFI_AI-main/outputs_finetune_layer4
Device: cuda
Epochs: 10
Output: /content/drive/MyDrive/LAAFI_AI-main/outputs_finetune_layer4


## 3. Charger les données

In [7]:
data_module = PCamDataModule(config.data)
train_loader, val_loader, test_loader = data_module.dataloaders()

print(f'Train: {len(train_loader.dataset):,} images')
print(f'Val:   {len(val_loader.dataset):,} images')

2026-07-06 23:43:30,398 | INFO | laafi_ai.data | Loading dataset 1aurent/PatchCamelyon
2026-07-06 23:43:31,937 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-06 23:43:32,036 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/1aurent/PatchCamelyon/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-07-06 23:43:32,114 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/1aurent/PatchCamelyon/e4bd149e7a868a9d811fdd9f9a9fb78c05c104ab/README.md "HTTP/1.1 200 OK"
2026-07-06 23:43:32,218 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/1aurent/PatchCamelyon/e4bd149e7a868a9d811fdd9f9a9fb78c05c104ab/README.md "HTTP/1.1 200 OK"


README.md:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

2026-07-06 23:43:32,892 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/1aurent/PatchCamelyon/resolve/e4bd149e7a868a9d811fdd9f9a9fb78c05c104ab/PatchCamelyon.py "HTTP/1.1 404 Not Found"
2026-07-06 23:43:33,203 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/1aurent/PatchCamelyon/1aurent/PatchCamelyon.py "HTTP/1.1 404 Not Found"
2026-07-06 23:43:33,374 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/1aurent/PatchCamelyon/revision/e4bd149e7a868a9d811fdd9f9a9fb78c05c104ab "HTTP/1.1 200 OK"
2026-07-06 23:43:33,477 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/1aurent/PatchCamelyon/resolve/e4bd149e7a868a9d811fdd9f9a9fb78c05c104ab/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-07-06 23:43:33,644 | INFO | httpx | HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=1aurent/PatchCamelyon "HTTP/1.1 200 OK"
2026-07-06 23:43:33,830 | INFO | httpx | HTTP Request:

data/train-00000-of-00013-4717c3cf92578c(…):   0%|          | 0.00/471M [00:00<?, ?B/s]

2026-07-06 23:43:41,357 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/1aurent/PatchCamelyon/resolve/e4bd149e7a868a9d811fdd9f9a9fb78c05c104ab/data/train-00001-of-00013-549914845b4273b1.parquet "HTTP/1.1 302 Found"


...

data/test-00001-of-00002-3bfa172e8818685(…):   0%|          | 0.00/375M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/262144 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/32768 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/32768 [00:00<?, ? examples/s]

Train: 262,144 images
Val:   32,768 images


## 4. Construire le modèle et le Trainer

In [8]:
model = build_resnet50_classifier(config.model)
print(f'Paramètres entraînables: {count_trainable_parameters(model):,}')

trainer = Trainer(model=model, config=config, device=device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s]


2026-07-06 23:47:14,786 | INFO | laafi_ai.model | Freezing ResNet50 backbone
2026-07-06 23:47:14,787 | INFO | laafi_ai.model | Unfreezing ResNet50 layer4
Paramètres entraînables: 14,966,785


/content/drive/MyDrive/LAAFI_AI-main/src/laafi_ai/trainer.py:100: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(


## 5. Reprendre depuis un checkpoint (si disponible)

Le Trainer cherche automatiquement le dernier fichier `.pt` dans le dossier `checkpoints/`.
- S'il en trouve un → il restaure les poids, l'optimizer, l'epoch, et l'historique.
- Sinon → il part de zéro.

In [9]:
resume_ckpt = trainer.resume_from_checkpoint(paths.checkpoint_dir)

if resume_ckpt is not None:
    print(f'\n✅ Checkpoint trouvé : epoch {resume_ckpt.epoch}, '
          f'best AUC = {resume_ckpt.best_metric:.4f}')
    print(f'   → Reprend à l\'epoch {resume_ckpt.next_epoch}')
else:
    print('\nAucun checkpoint trouvé — entraînement from scratch.')

2026-07-06 23:47:15,894 | INFO | laafi_ai.checkpoints | Latest checkpoint found: /content/drive/MyDrive/LAAFI_AI-main/outputs_finetune_layer4/checkpoints/epoch_006.pt
2026-07-06 23:47:19,419 | INFO | laafi_ai.checkpoints | Checkpoint loaded ← /content/drive/MyDrive/LAAFI_AI-main/outputs_finetune_layer4/checkpoints/epoch_006.pt  (epoch 6, best_metric=0.9584)

✅ Checkpoint trouvé : epoch 6, best AUC = 0.9584
   → Reprend à l'epoch 7


## 6. Lancer l'entraînement

In [ ]:
history = trainer.fit(train_loader, val_loader, resume_checkpoint=resume_ckpt)

2026-07-06 23:47:19,432 | WARNING | laafi_ai.trainer | mlflow non installé. Désactiver use_mlflow ou pip install mlflow.
2026-07-06 23:47:19,433 | WARNING | laafi_ai.trainer | Fallback: entraînement sans MLflow.


train epoch 1:   0%|          | 0/8192 [00:00<?, ?it/s]

/content/drive/MyDrive/LAAFI_AI-main/src/laafi_ai/trainer.py:301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


eval:   0%|          | 0/1024 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd33d543740>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7dd33d543740>^
Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^    ^^self._shutdown_workers()
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^^    ^if w.is_alive():^
 ^  ^ ^^ ^ ^ ^^^^^^^

...

/content/drive/MyDrive/LAAFI_AI-main/src/laafi_ai/trainer.py:301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


eval:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-07 03:17:24,082 | INFO | laafi_ai.trainer | Epoch 8 summary: {'epoch': 8.0, 'train_loss': 0.07437781023143941, 'val_loss': 0.46184367930709413, 'val_auc': 0.9430505435406129, 'val_accuracy': 0.86981201171875, 'val_sensitivity': 0.7785448103121755, 'val_specificity': 0.9609122507469968, 'val_precision': 0.9521105715353008, 'val_average_precision': 0.9509511065502985}
2026-07-07 03:17:30,528 | INFO | laafi_ai.checkpoints | Checkpoint saved → outputs_finetune_layer4/checkpoints/epoch_008.pt  (epoch 8, best_metric=0.9592)


train epoch 9:   0%|          | 0/8192 [00:00<?, ?it/s]

/content/drive/MyDrive/LAAFI_AI-main/src/laafi_ai/trainer.py:301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd33d543740>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd33d543740>
Traceback (most recent call last):

## 7. Résumé de l'entraînement

In [ ]:
import matplotlib.pyplot as plt

if history:
    epochs = [r['epoch'] for r in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, [r['train_loss'] for r in history], label='Train loss')
    ax1.plot(epochs, [r['val_loss'] for r in history], label='Val loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.set_title('Loss')
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, [r['val_auc'] for r in history], label='Val AUC', marker='o')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('AUC')
    ax2.legend()
    ax2.set_title('Validation AUC')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(paths.figures_dir / 'training_curves.png', dpi=150)
    plt.show()
    print('Courbes sauvegardées dans', paths.figures_dir / 'training_curves.png')
else:
    print('Aucune epoch exécutée (probablement déjà terminé).')

---
**Prochain notebook** : `03_eval.ipynb` pour évaluer le meilleur checkpoint sur le test set.